# 3201 Hashing Assignment

In [31]:
import csv
import string
import time
import random
from Crypto.Hash import SHA256
from bcrypt import checkpw, hashpw
import nltk
nltk.download('words')
from nltk.corpus import words

[nltk_data] Downloading package words to
[nltk_data]     /Users/diegomelgoza/nltk_data...
[nltk_data]   Package words is already up-to-date!


## Task 1

Task 1.1

In [19]:
def hash_something(input: str):
    return SHA256.new(input.encode("utf-8")).hexdigest()

In [ ]:
input = "Something"

In [ ]:
print(f"Final hash: {hash_something(input)}")

Task 1.2

In [20]:
def get_hamming_distance(str1:str, str2:str) -> int:
    str1_int = int.from_bytes(str1.encode("utf-8"))
    str2_int = int.from_bytes(str2.encode("utf-8"))
    and_res = str1_int ^ str2_int
    return f"{and_res:b}".count("1")

In [ ]:
str1 = "cat"
str2 = "bat"
get_hamming_distance(str1, str2)

In [ ]:
print(f"{hash_something(str1)}")
print(f"{hash_something(str2)}")

In [ ]:
str1 = "abcdefgh"
str2 = "abbdefgh"
get_hamming_distance(str1, str2)

In [ ]:
print(f"{hash_something(str1)}")
print(f"{hash_something(str2)}")

Task 1.3

In [21]:
#create a random string using the ASCII letter set
#get each individual character by randomly selecting with Random
def random_string() -> str:
    letters = list(string.ascii_letters)
    text = ''
    for i in range(10):
        text = text + letters[random.randint(0, len(letters) - 1)]
    return text

In [22]:
#Create a new hash using encoded text
#then truncate it by
#directly extracting the needed # of bits from the bitstring
def get_truncated_hash(bits: int, text: str) -> str:
    new_hash = SHA256.new(text.encode("utf-8"))
    truncated = f"{int.from_bytes(new_hash.digest()):b}"[:bits]
    return truncated

In [32]:
#find a hash collision in a hash space "bits" # of bits
#use a dictionary to keep track of hashes as they are
#randomly generated. function with either find a collision
#or give up after the defined # of max attempts.
def find_hash_collision(bits, max_attempts):
    seen = {}
    start = time.time()
    for att in range(max_attempts):
        text = random_string()
        t_hash = get_truncated_hash(bits, text)
        if t_hash in seen:
            stop = time.time() - start
            return seen[t_hash], text, t_hash, stop, att
        else:
            seen[t_hash] = text
    stop = time.time() - start
    return None, None, None, stop, max_attempts

In [33]:
#Hash collision experiment that returns the collision values,
#the time to get a collision, and the # of hashes generated
#for every even # of bits from 8 to 50.
def hash_collision_experiment():
    results = {}
    for bits in range(8, 52, 2):
        text1, text2, hash_res, time, inputs = find_hash_collision(bits, 10000000000)
        results[bits] = [text1, text2, hash_res, time, inputs]
    with open("./hash_collision_results.txt", "w") as f:
        f.write("bits: text1, text2, hash_res, time, inputs\n")
        for bits in results:
            f.write(f"{bits}: {results[bits]}\n")


In [34]:
hash_collision_experiment()

# Part 2

In [2]:
def crack_password(pw_hash:str, word_corpus:list[str]):
    #find the word in the wordset that returns the desired hash
    for word in word_corpus:
        #checkpw from bcrypt library
        if checkpw(word.encode("utf-8"), pw_hash.encode("utf-8")):
            return word
    return None

In [3]:
def get_all_passwords(filepath:str = 'user_hashes.txt'):
    #word corpus from nltk words library
    word_corpus = [word for word in words.words() if 10 >= len(word) >= 6]
    user_hashes = {}
    user_passwords = {}
    #get every hash within the hashes file
    with open(filepath) as f:
        for line in f:
            user, hashval = line.split(":")
            user_hashes[user] = hashval

    #crack every password and write it to the temporary dictionary
    for user in user_hashes:
        password = crack_password(user_hashes[user], word_corpus)
        if password:
            user_passwords[user] = password
        else:
            user_passwords[user] = "passworderror!!!"

    #write all the passwords to the file.
    with open("./user_passwords.txt", "w+") as f:
        f.write("user: password")
        for user in user_passwords:
            f.write(f"{user}: {user_passwords[user]}\n")
